# 02 - Inference on the trained model

Loads the published `best.pt` from the GitHub Release and runs it on validation images and on new
images the model has never seen. Both the weights and the images come from this repository, so the
notebook runs end to end on a clean runtime with nothing read from a local drive.

**How to run:** Runtime -> **Run all**. A GPU is nice but not required.

**Expected runtime:** ~2-3 min including the install.

> This model is an assistive tool for preliminary screening only. It produces false negatives and
> must not be used as the sole verifier for life-safety decisions. A qualified reviewer checks
> every detection.

## 1. Setup

In [ ]:
!pip install -q ultralytics==8.4.155

import torch, ultralytics
ultralytics.checks()
print("ultralytics:", ultralytics.__version__, "| cuda:", torch.cuda.is_available())

## 2. Configuration

Weights come from a public URL, never from a personal drive — that is what makes this notebook runnable by a stranger.

In [ ]:
REPO = "caprijopi-alt/ppe-detection-yolov8"
BRANCH = "main"

WEIGHTS_URL = f"https://github.com/{REPO}/releases/download/v1.0/best.pt"
RAW = f"https://raw.githubusercontent.com/{REPO}/{BRANCH}"
API = f"https://api.github.com/repos/{REPO}/contents"

# classes: helmet, no-helmet, no-vest, person, vest
CONF_THRESHOLD = 0.35    # 0.35 matches the training notebook; lower it to trade precision for recall
                         # on no-helmet (precision 0.947 leaves headroom) - see docs/error_analysis.md
IMGSZ = 640

In [ ]:
import os, urllib.request

if not os.path.exists("/content/best.pt"):
    urllib.request.urlretrieve(WEIGHTS_URL, "/content/best.pt")
print(round(os.path.getsize("/content/best.pt")/1e6, 2), "MB downloaded")

from ultralytics import YOLO
model = YOLO("/content/best.pt")
print("classes:", model.names)

In [ ]:
# Images are pulled from the repo itself, so this notebook runs end to end with no uploads.
import json, os, glob, urllib.request

def fetch_samples(folder, dest):
    """Download every image in samples/<folder>/ of the repo into dest. Returns the local paths."""
    os.makedirs(dest, exist_ok=True)
    with urllib.request.urlopen(f"{API}/samples/{folder}?ref={BRANCH}") as r:
        entries = json.load(r)
    paths = []
    for e in entries:
        if e["type"] != "file" or not e["name"].lower().endswith((".jpg", ".jpeg", ".png")):
            continue
        local = os.path.join(dest, e["name"])
        if not os.path.exists(local):
            urllib.request.urlretrieve(e["download_url"], local)
        paths.append(local)
    return sorted(paths)

## 3. Validation images

A sanity check that the released weights behave as reported. These images were held out of training but the model was tuned against them — section 4 is the honest test.

In [ ]:
from IPython.display import Image, display

val_images = fetch_samples("val", "/content/val_images")
print(len(val_images), "validation images fetched from the repo")

res = model.predict(val_images, conf=CONF_THRESHOLD, imgsz=IMGSZ, save=True,
                    project="/content/runs", name="val_preds")

for p in sorted(glob.glob(f"{res[0].save_dir}/*"))[:5]:
    display(Image(filename=p, width=560))

## 4. New images

The real demonstration: photographs from outside the dataset. If it works here, there is a product.

In [ ]:
new_images = fetch_samples("new", "/content/new_images")
print(len(new_images), "unseen images fetched from the repo")

res_new = model.predict(new_images, conf=CONF_THRESHOLD, imgsz=IMGSZ, save=True,
                        project="/content/runs", name="new_preds")

for r, p in zip(res_new, sorted(glob.glob(f"{res_new[0].save_dir}/*"))):
    counts = {}
    for c in r.boxes.cls.tolist():
        counts[model.names[int(c)]] = counts.get(model.names[int(c)], 0) + 1
    print(os.path.basename(p), "->", counts or "no detections")
    display(Image(filename=p, width=560))

## 5. Confidence sweep

The threshold is a business decision, not a default. This shows what recall-first vs precision-first actually costs on your own images.

In [ ]:
for t in (0.10, 0.25, 0.50, 0.75):
    r = model.predict(new_images, conf=t, imgsz=IMGSZ, verbose=False)
    total = sum(len(x.boxes) for x in r)
    print(f"conf {t:>4}: {total:>3} detections across {len(new_images)} images")

## 6. Download the evidence

In [ ]:
import shutil
shutil.make_archive("/content/inference_evidence", "zip", "/content/runs")
print("written: /content/inference_evidence.zip")

try:
    from google.colab import files
    files.download("/content/inference_evidence.zip")
except Exception as e:
    print("(not in Colab, skipping the browser download)", e)

## How to read these results

- A missed instance is a **false negative** — the expensive error for safety screening.
- A box on nothing is a **false positive** — the expensive error when detections trigger cost.
- Right object with the wrong label is **class confusion**; right place with a sloppy box is a
  **localisation error**.

Three of each go in [`docs/error_analysis.md`](../docs/error_analysis.md) with a hypothesis for
*why*. A well-explained failure is worth more than a hidden one.